<a href="https://colab.research.google.com/github/sarmanoveduard-design/intelligent-knowledge-base/blob/main/generation-benchmark/notebooks/generation_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOST — Generation Benchmark Colab

Готовый воспроизводимый тест на одном frozen Top‑10 контексте и одном prompt.

**MAIN_119_APPROVED** — 119 подтверждённых вопросов.  
**EXTENDED_149_ALL** — зарезервирован под все 149 вопросов (119 approved + 30 provisional), будет включён после подготовки отдельного frozen snapshot.

Модели:
- `qwen3:8b` — локально через Ollama, `think=false`
- `gpt-5.6-luna` — OpenAI Responses API, `reasoning.effort="none"`
- `gpt-5.6-terra` — OpenAI Responses API, `reasoning.effort="none"`

Для MAIN benchmark-файлы скачиваются **автоматически из GitHub** и привязаны к commit `3442a97`.
API-ключи в notebook и GitHub не хранятся.

In [6]:
!pip -q install "openai>=2.0,<3" requests pandas


In [7]:
#@title Настройки
BENCHMARK_SET = "MAIN_119_APPROVED" #@param ["MAIN_119_APPROVED", "EXTENDED_149_ALL"]
MODELS_TO_RUN = "all3" #@param ["qwen3:8b", "gpt-5.6-luna", "gpt-5.6-terra", "all3"]
RUN_SCOPE = "one" #@param ["one", "all"]
QUERY_NUMBER = 87 #@param {type:"integer"}
CONFIRM_FULL_RUN = False #@param {type:"boolean"}

EXPECTED_INPUT_SHA256 = "1b94bfca536b36e386d663572f72e9abe91b19da725d83b18df040277fd52f93"
EXPECTED_PROMPT_SHA256 = "f05cb6f1ec4e05d20aeb1cf7bfc8c9d2e508450b42073698707707a6de923a6c"
PINNED_BENCHMARK_COMMIT = "3442a97b81f0a12d884b09d0b1be550a539b40e5"

if BENCHMARK_SET == "EXTENDED_149_ALL":
    raise RuntimeError("EXTENDED_149_ALL пока не активирован: frozen Top-10 snapshot для 149 вопросов ещё не подготовлен.")

if RUN_SCOPE == "all" and not CONFIRM_FULL_RUN:
    raise RuntimeError("Для полного прогона поставьте CONFIRM_FULL_RUN=True")

SELECTED_MODELS = (
    ["qwen3:8b", "gpt-5.6-luna", "gpt-5.6-terra"]
    if MODELS_TO_RUN == "all3" else [MODELS_TO_RUN]
)

print("BENCHMARK_SET =", BENCHMARK_SET)
print("MODELS =", SELECTED_MODELS)
print("RUN_SCOPE =", RUN_SCOPE)
print("QUERY_NUMBER =", QUERY_NUMBER if RUN_SCOPE == "one" else "не используется")


BENCHMARK_SET = MAIN_119_APPROVED
MODELS = ['qwen3:8b', 'gpt-5.6-luna', 'gpt-5.6-terra']
RUN_SCOPE = one
QUERY_NUMBER = 87


In [8]:
#@title Автоматически скачать frozen benchmark из GitHub
from pathlib import Path
import hashlib, json, requests

BASE = "https://raw.githubusercontent.com/sarmanoveduard-design/intelligent-knowledge-base/" + PINNED_BENCHMARK_COMMIT + "/benchmarks/generation/main_119"
INPUT_PATH = Path("/content/model_input_top10.json")
PROMPT_PATH = Path("/content/generation_v1_benchmark.txt")

def download(url, path):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    path.write_bytes(r.content)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

download(BASE + "/model_input_top10.json", INPUT_PATH)
download(BASE + "/generation_v1_benchmark.txt", PROMPT_PATH)

input_hash = sha256_file(INPUT_PATH)
prompt_hash = sha256_file(PROMPT_PATH)

if input_hash != EXPECTED_INPUT_SHA256:
    raise RuntimeError(f"MODEL INPUT SHA mismatch: {input_hash}")
if prompt_hash != EXPECTED_PROMPT_SHA256:
    raise RuntimeError(f"PROMPT SHA mismatch: {prompt_hash}")

dataset = json.loads(INPUT_PATH.read_text(encoding="utf-8-sig"))
prompt = PROMPT_PATH.read_text(encoding="utf-8-sig").strip()

if len(dataset.get("queries", [])) != 119:
    raise RuntimeError("Ожидалось 119 вопросов")
if dataset.get("context_top_k") != 10:
    raise RuntimeError("Ожидался context_top_k=10")

for case in dataset["queries"]:
    qn = int(case["query_number"])
    if not str(case.get("query", "")).strip():
        raise RuntimeError(f"Пустой вопрос: {qn}")
    if len(case.get("contexts", [])) != 10:
        raise RuntimeError(f"У вопроса {qn} не 10 контекстов")
    if "answer_hint" in json.dumps(case, ensure_ascii=False):
        raise RuntimeError(f"STOP: answer_hint leaked into query {qn}")

print("BENCHMARK INPUT: OK")
print("QUESTIONS=119")
print("CONTEXT_TOP_K=10")
print("MODEL_INPUT_SHA256=", input_hash)
print("PROMPT_SHA256=", prompt_hash)


BENCHMARK INPUT: OK
QUESTIONS=119
CONTEXT_TOP_K=10
MODEL_INPUT_SHA256= 1b94bfca536b36e386d663572f72e9abe91b19da725d83b18df040277fd52f93
PROMPT_SHA256= f05cb6f1ec4e05d20aeb1cf7bfc8c9d2e508450b42073698707707a6de923a6c


In [9]:
#@title Подготовить OpenAI key и Qwen/Ollama
import getpass, os, requests, shutil, subprocess, time, platform

needs_openai = any(m.startswith("gpt-") for m in SELECTED_MODELS)

if needs_openai and not os.environ.get("OPENAI_API_KEY"):
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
    if not key:
        key = getpass.getpass("OPENAI_API_KEY: " ).strip()
    if not key:
        raise RuntimeError("OPENAI_API_KEY не задан")
    os.environ["OPENAI_API_KEY"] = key

if "qwen3:8b" in SELECTED_MODELS:
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)
    else:
        print("WARNING: GPU не обнаружен. Qwen будет очень медленным.")

    if not shutil.which("ollama"):
        machine = platform.machine().lower()
        if machine in ("x86_64", "amd64"):
            package = "ollama-linux-amd64.tar.zst"
        elif machine in ("aarch64", "arm64"):
            package = "ollama-linux-arm64.tar.zst"
        else:
            raise RuntimeError(f"Неподдерживаемая архитектура для Ollama: {machine}")

        subprocess.run(
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq curl zstd"],
            check=True,
        )
        subprocess.run(
            [
                "bash",
                "-lc",
                f"curl -fsSL https://ollama.com/download/{package} | tar --zstd -x -C /usr",
            ],
            check=True,
        )

    def ready():
        try:
            return requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok
        except Exception:
            return False

    if not ready():
        log = open("/tmp/ollama-colab.log", "w")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=log,
            stderr=subprocess.STDOUT,
            env=os.environ.copy(),
        )
        for _ in range(60):
            if ready():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama не запустился. Проверьте /tmp/ollama-colab.log")

    subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)

print("ENVIRONMENT: READY")


ENVIRONMENT: READY


In [10]:
#@title Общий runner
import math, time, json
import pandas as pd, requests
from openai import OpenAI

def pct(values, f):
    if not values: return None
    a=sorted(values)
    return a[max(0, math.ceil(len(a)*f)-1)]

def user_message(case):
    p=["ВОПРОС ПОЛЬЗОВАТЕЛЯ:",case["query"],"","ПРЕДОСТАВЛЕННЫЙ КОНТЕКСТ:"]
    for ctx in case["contexts"]:
        p += ["",f"[Источник {ctx['rank']}]",f"document_id: {ctx['document_id']}",
              f"section_ref: {ctx['section_ref']}","text:",ctx["text"]]
    return "\n".join(p)

client = OpenAI() if needs_openai else None

def call_model(model, case):
    started=time.perf_counter()
    if model=="qwen3:8b":
        r=requests.post("http://127.0.0.1:11434/api/chat",json={
            "model":model,
            "messages":[{"role":"system","content":prompt},{"role":"user","content":user_message(case)}],
            "think":False,"stream":False,"keep_alive":"30m",
            "options":{"num_ctx":8192,"num_predict":512},
        },timeout=600)
        r.raise_for_status()
        raw=r.json()
        answer=(raw.get("message") or {}).get("content","").strip()
        if not answer: raise RuntimeError("Qwen returned empty answer")
        return {"answer":answer,"latency_seconds":time.perf_counter()-started,
                "input_tokens":raw.get("prompt_eval_count"),"output_tokens":raw.get("eval_count")}
    resp=client.responses.create(model=model,reasoning={"effort":"none"},instructions=prompt,
                                 input=user_message(case),max_output_tokens=512,store=False)
    answer=resp.output_text.strip()
    if not answer: raise RuntimeError(f"{model} returned empty answer")
    u=resp.usage
    return {"answer":answer,"latency_seconds":time.perf_counter()-started,
            "input_tokens":u.input_tokens if u else None,"output_tokens":u.output_tokens if u else None}

def run_model(model,cases):
    rows=[]
    for i,case in enumerate(cases,1):
        qn=int(case["query_number"])
        try:
            x=call_model(model,case)
            row={"status":"ok","model":model,"query_number":qn,"query":case["query"],**x}
            print(f"[{model}] [{i:03d}/{len(cases):03d}] q={qn:03d} OK {x['latency_seconds']:.2f}s")
        except Exception as e:
            row={"status":"error","model":model,"query_number":qn,"query":case["query"],
                 "error_type":type(e).__name__,"error_message":str(e)[:300]}
            print(f"[{model}] [{i:03d}/{len(cases):03d}] q={qn:03d} ERROR {type(e).__name__}")
        rows.append(row)
    ok=[r for r in rows if r["status"]=="ok"]
    lat=[r["latency_seconds"] for r in ok]
    summary={"model":model,"queries":len(cases),"success":len(ok),"errors":len(rows)-len(ok),
             "p50_latency_seconds":pct(lat,.5),"p95_latency_seconds":pct(lat,.95),
             "input_tokens":sum(int(r.get("input_tokens") or 0) for r in ok),
             "output_tokens":sum(int(r.get("output_tokens") or 0) for r in ok)}
    return rows,summary


In [11]:
#@title Запуск и вывод результатов
if RUN_SCOPE=="one":
    cases=[q for q in dataset["queries"] if int(q["query_number"])==int(QUERY_NUMBER)]
    if len(cases)!=1: raise RuntimeError("query_number не найден ровно один раз")
else:
    cases=list(dataset["queries"])

all_rows=[]
summaries=[]

for model in SELECTED_MODELS:
    print("\n===",model,"===")
    rows,summary=run_model(model,cases)
    all_rows.extend(rows)
    summaries.append(summary)

print("\n=== SUMMARY ===")
display(pd.DataFrame(summaries))

print("\n=== ANSWERS ===")
display(pd.DataFrame(all_rows)[["model","query_number","query","status","latency_seconds","answer"]])



=== qwen3:8b ===
[qwen3:8b] [001/001] q=087 OK 117.79s

=== gpt-5.6-luna ===
[gpt-5.6-luna] [001/001] q=087 OK 5.06s

=== gpt-5.6-terra ===
[gpt-5.6-terra] [001/001] q=087 OK 2.72s

=== SUMMARY ===


,model,queries,success,errors,p50_latency_seconds,p95_latency_seconds,input_tokens,output_tokens
0,qwen3:8b,1,1,0,117.793247,117.793247,6417,67
1,gpt-5.6-luna,1,1,0,5.056521,5.056521,4825,171
2,gpt-5.6-terra,1,1,0,2.722152,2.722152,4825,119



=== ANSWERS ===


,model,query_number,query,status,latency_seconds,answer
0,qwen3:8b,87,В программу корпоративного здоровья хотим вклю...,ok,117.793247,"Факторы среды обитания, к которым относятся во..."
1,gpt-5.6-luna,87,В программу корпоративного здоровья хотим вклю...,ok,5.056521,К таким воздействиям относятся следующие групп...
2,gpt-5.6-terra,87,В программу корпоративного здоровья хотим вклю...,ok,2.722152,- Защита от инфекций относится к **биологическ...


### Как запускать

Для быстрой проверки:  
`MODELS_TO_RUN=all3`, `RUN_SCOPE=one`, `QUERY_NUMBER=87`.

Для полного MAIN benchmark:  
`MODELS_TO_RUN=all3`, `RUN_SCOPE=all`, `CONFIRM_FULL_RUN=True`.

Для OpenAI используется ваш собственный `OPENAI_API_KEY` через Colab Secrets или скрытый ввод.